Combien de contrats actifs par ville et par produit ?

In [0]:
%sql
SELECT city, product, COUNT(*) AS active_contracts
FROM alpenia_dev.bronze.brz_contracts
WHERE status = 'active'
GROUP BY city, product
ORDER BY city, product;


Ce tableau montre la répartition du portefeuille actif entre villes et produits ; les combinaisons les plus fournies indiquent où se concentre l’activité d’Alpenia

Quel est le nombre de sinistres, le montant total réglé et le montant moyen par sinistre, par ville ?

In [0]:
%sql
SELECT city, COUNT(*) AS perils, ROUND(SUM(amount_eur), 2) AS total_amount_eur, ROUND(AVG(amount_eur), 2) AS average_amount_eur 
FROM alpenia_dev.bronze.brz_claims ct
JOIN alpenia_dev.bronze.brz_contracts cn ON ct.contract_id = cn.contract_id 
GROUP BY city
;

Le tableau parle pour lui-même

Quel est le délai moyen de règlement, en jours, par nature de sinistre ?

In [0]:
%sql
SELECT peril, IFNULL(ROUND(AVG(date_diff(settled_ts, declared_ts)), 0), -1) AS avg_days
FROM alpenia_dev.bronze.brz_claims
GROUP BY peril;

Entre 50 et 51 jours en moyenne. On sent au vu de cette faible dispersion que les règlements s'effectuent selon une procédure fixe et maîtrisée. 

Quels sont les 10 clients ayant abandonné le plus de devis dans les interactions web ?

In [0]:
%sql
SELECT client_id, COUNT(*) AS abandoned_quotes
FROM alpenia_dev.bronze.brz_web_events
WHERE action = 'quote_abandoned' 
  AND client_id IS NOT NULL
GROUP BY client_id
ORDER BY abandoned_quotes DESC, client_id ASC
LIMIT 10;

Ces dix clients concentrent le plus d’abandons de devis : ce sont les cibles prioritaires pour une relance commerciale ou une analyse des freins dans le parcours de souscription.

Quel est le coût total des expertises par type de constat, et combien d'expertises sont encore ouvertes ?

In [0]:
%sql

SELECT
  finding_type, 
  ROUND(SUM(cost_eur), 2) AS total_cost_eur, 
  COUNT(*) AS expertises, 
  SUM(CASE WHEN is_open THEN 1 ELSE 0 END) AS open_expertises
FROM alpenia_dev.bronze.brz_expertises
GROUP BY finding_type
ORDER BY total_cost_eur DESC;

Ce tableau identifie les types de constat qui pèsent le plus dans le coût d’expertise, et le nombre de dossiers encore ouverts indique le travail restant à clôturer, donc un coût susceptible d’augmenter.

In [0]:
%sql
DESCRIBE DETAIL alpenia_dev.bronze.brz_contracts;


In [0]:
%sql
DESCRIBE HISTORY alpenia_dev.bronze.brz_contracts;